# THPTQG Trends — Google Colab

Phân tích điểm THPT quốc gia **2021–2025** và dự báo xu hướng.

**Chạy lần lượt từ trên xuống.** File CSV ~805MB không có trên GitHub — dùng **Google Drive** hoặc upload `cleaned_data.csv`.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels pyarrow


In [ ]:
import os, sys
from pathlib import Path

ROOT = Path("/content/thptqg-trends")
if not ROOT.exists():
    !git clone https://github.com/2274802010922/thptqg-trends.git /content/thptqg-trends
else:
    !cd /content/thptqg-trends && git pull

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.display import setup_display
setup_display()
print("Project:", ROOT)


## Nguồn dữ liệu (chọn **một** cách)

### Cách A: Google Drive (khuyên dùng)


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
CSV_PATH = "/content/drive/MyDrive/do an thuc tap/cleaned_data.csv"
assert Path(CSV_PATH).exists(), f"Không tìm thấy: {CSV_PATH}"
print("CSV:", CSV_PATH)


### Cách B: Upload trực tiếp


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = "/content/cleaned_data.csv"


In [ ]:
from src.config import configure

YEAR_MIN, YEAR_MAX = 2021, 2025
configure(csv_path=CSV_PATH, year_min=YEAR_MIN, year_max=YEAR_MAX)
print(f"Phạm vi: {YEAR_MIN}-{YEAR_MAX}")


In [ ]:
import time
from IPython.display import display
from src.load_data import count_rows
from src.aggregates import save_aggregates
from src.forecast import run_forecast_pipeline
from src.plots import generate_all_figures
from src.report import generate_report
from src.display import pretty_counts, pretty_forecast

t0 = time.time()
print("1/5 Kiểm tra dữ liệu")
display(pretty_counts(count_rows()))
print("2/5 Aggregate")
save_aggregates()
print("3/5 Dự báo")
display(pretty_forecast(run_forecast_pipeline()))
print("4/5 Biểu đồ")
for f in generate_all_figures():
    print(" ", f.name)
print("5/5 Báo cáo:", generate_report())
print(f"HOÀN TẤT {(time.time()-t0)/60:.1f} phút")


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown(Path("reports/BAO_CAO.md").read_text(encoding="utf-8")))


In [ ]:
%matplotlib inline
from IPython.display import Image, display
from pathlib import Path
for p in sorted(Path("outputs/figures").glob("*.png")):
    print(p.name)
    display(Image(filename=str(p)))


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

zip_path = Path("/content/thptqg_full.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["outputs", "reports"]:
        for f in Path(folder).rglob("*"):
            if f.is_file():
                z.write(f, f.as_posix())
files.download(str(zip_path))
